# Notebook 1 — Read & Join the Tables

## Goal
Read the Olist tables from PostgreSQL, understand their structure and relationships, aggregate tables with multiple rows per order, and create one ML table with one row per order.

In [14]:
%pip install pandas sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [65]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

## 1. Connect to PostgreSQL

The data is stored in a local PostgreSQL database.

We first create a database connection and test it before reading the tables.

In [16]:
DB_URL = (
    "postgresql+psycopg2://postgres:postgres123"
    "@localhost:5432/olist_brazilian_ecommerce"
)

engine = create_engine(DB_URL)

with engine.connect() as connection:
    print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [17]:
pd.read_sql("SELECT 1;", engine)

,?column?
0,1


## 2. Inspect the database tables

First, we check the available tables, their row counts, columns, and a small sample.

This gives us a quick understanding of what each table contains before joining them.

In [18]:
tables = pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""", engine)

tables

,table_name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


In [19]:
table_names = tables["table_name"].tolist()

for table in table_names:
    sample = pd.read_sql(
        f"SELECT * FROM {table} LIMIT 3;",
        engine
    )

    row_count = pd.read_sql(
        f"SELECT COUNT(*) AS row_count FROM {table};",
        engine
    ).iloc[0]["row_count"]

    print("\n" + "=" * 60)
    print(f"Table: {table}")
    print(f"Rows: {row_count}")
    print(f"Columns: {list(sample.columns)}")
    print("=" * 60)

    display(sample)


Table: customers
Rows: 99441
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



Table: geolocation
Rows: 1000163
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545622,-46.639294,sao paulo,SP
1,1046,-23.546082,-46.644820,sao paulo,SP
2,1046,-23.546130,-46.642952,sao paulo,SP



Table: order_items
Rows: 112650
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


...

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto



Table: products
Rows: 32951
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.0,18.0,9.0,15.0



Table: sellers
Rows: 3095
Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


## 3. Understand the Table Structure

Each table has a different level of detail.

For example, one row in `orders` represents an order, while one order can
have several rows in `order_items` or `order_payments`.

We need to understand this before joining the tables.

In [20]:
table_grain = pd.DataFrame({
    "table": [
        "orders",
        "customers",
        "order_items",
        "sellers",
        "products",
        "geolocation",
        "order_payments",
        "order_reviews",
        "product_category_name_translation"
    ],
    "one_row_represents": [
        "One order",
        "One customer",
        "One item in an order",
        "One seller",
        "One product",
        "One geographic record",
        "One payment record",
        "One review",
        "One product category"
    ]
})

display(table_grain)

,table,one_row_represents
0,orders,One order
1,customers,One customer
2,order_items,One item in an order
3,sellers,One seller
4,products,One product
5,geolocation,One geographic record
6,order_payments,One payment record
7,order_reviews,One review
8,product_category_name_translation,One product category


## 4. Check Important Keys

Before joining the tables, we check the main keys.

Some keys should be unique, while others are expected to repeat because
one order can have multiple items or payments.

In [21]:
orders_key = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_order_ids
    FROM orders;
    """,
    engine
)

items_key = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (order_id, order_item_id)) AS unique_item_keys
    FROM order_items;
    """,
    engine
)

payments_key = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (order_id, payment_sequential)) AS unique_payment_keys
    FROM order_payments;
    """,
    engine
)

print("Orders:")
display(orders_key)

print("Order items:")
display(items_key)

print("Payments:")
display(payments_key)

Orders:


,total_rows,unique_order_ids
0,99441,99441


Order items:


,total_rows,unique_item_keys
0,112650,112650


Payments:


,total_rows,unique_payment_keys
0,103886,103886


## 5. Read the orders table

The orders table is the main table because the prediction is made at the order level.

We will use it as the base of the final ML table.

In [22]:
orders = pd.read_sql(
    """
    SELECT
        order_id,
        customer_id,
        order_status,
        order_purchase_timestamp,
        order_approved_at,
        order_delivered_carrier_date,
        order_delivered_customer_date,
        order_estimated_delivery_date
    FROM orders;
    """,
    engine
)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [23]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

Rows: 99441
Unique orders: 99441
Duplicate order IDs: 0


## 6. Read Customers

Each order is connected to a customer through `customer_id`.

We keep the customer information that may be useful later, including the
customer location fields.

In [24]:
customers = pd.read_sql(
    """
    SELECT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state
    FROM customers;
    """,
    engine
)

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [25]:
print("Rows:", len(customers))
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Duplicate customer IDs:", customers["customer_id"].duplicated().sum())

Rows: 99441
Unique customer IDs: 99441
Duplicate customer IDs: 0


## 7. Read Order Items

An order can contain more than one item, so `order_items` can have several
rows for the same order.

We will first connect the items to their products and sellers, then aggregate
the information to one row per order.

In [26]:
order_items = pd.read_sql(
    """
    SELECT
        order_id,
        order_item_id,
        product_id,
        seller_id,
        shipping_limit_date,
        price,
        freight_value
    FROM order_items;
    """,
    engine
)

order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [27]:
print("Rows:", len(order_items))
print("Unique orders:", order_items["order_id"].nunique())

Rows: 112650
Unique orders: 98666


## 8. Read Products

We only keep the product information that can help describe the physical
size or type of the order.

The product name length, description length, and number of photos will not
be used as features.

In [28]:
products = pd.read_sql(
    """
    SELECT
        product_id,
        product_category_name,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm
    FROM products;
    """,
    engine
)

products.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0


In [29]:
print("Rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())
print("Duplicate product IDs:", products["product_id"].duplicated().sum())

Rows: 32951
Unique product IDs: 32951
Duplicate product IDs: 0


## 9. Join Order Items with Products

Each item is linked to a product through `product_id`.

This join is still at the item level. We will aggregate it later at the
order level.

In [30]:
items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Rows after join:", len(items_with_products))

Rows after join: 112650


## 10. Calculate Product Volume

The product dimensions can be combined into one value that represents the
approximate product volume.

We will later add the volumes of the products inside each order.

In [31]:
items_with_products["product_volume_cm3"] = (
    items_with_products["product_length_cm"]
    * items_with_products["product_height_cm"]
    * items_with_products["product_width_cm"]
)

## 11. Aggregate Item Information

Now we convert the item-level data to order-level data.

For each order, we calculate:

- Number of items
- Number of sellers
- Total price
- Total freight
- Total product weight
- Total product volume
- Number of product categories

In [32]:
items_agg = (
    items_with_products
    .groupby("order_id")
    .agg(
        number_of_items=("order_item_id", "count"),
        number_of_sellers=("seller_id", "nunique"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        total_product_weight=("product_weight_g", "sum"),
        total_product_volume=("product_volume_cm3", "sum"),
        number_of_product_categories=("product_category_name", "nunique")
    )
    .reset_index()
)

items_agg.head()

,order_id,number_of_items,number_of_sellers,total_price,total_freight,total_product_weight,total_product_volume,number_of_product_categories
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,58.90,13.29,650.0,3528.0,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,239.90,19.93,30000.0,60000.0,1
2,000229ec398224ef6ca0657da4fc703e,1,1,199.00,17.87,3050.0,14157.0,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,12.99,12.79,200.0,2400.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,199.90,18.14,3750.0,42000.0,1


In [33]:
print("Rows after aggregation:", len(items_agg))
print("Unique orders:", items_agg["order_id"].nunique())
print("Duplicate order IDs:", items_agg["order_id"].duplicated().sum())

Rows after aggregation: 98666
Unique orders: 98666
Duplicate order IDs: 0


## 12. Read Sellers

A seller can appear in many order items.

We will use seller information to get the seller location and compare it
with the customer location.

In [34]:
sellers = pd.read_sql(
    """
    SELECT
        seller_id,
        seller_zip_code_prefix,
        seller_city,
        seller_state
    FROM sellers;
    """,
    engine
)

sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [35]:
print("Rows:", len(sellers))
print("Unique seller IDs:", sellers["seller_id"].nunique())
print("Duplicate seller IDs:", sellers["seller_id"].duplicated().sum())

Rows: 3095
Unique seller IDs: 3095
Duplicate seller IDs: 0


## 13. Read Geolocation

The geolocation table can contain several records for the same ZIP prefix.

Because of that, we first create one representative location for each
ZIP prefix by taking the average latitude and longitude.

These coordinates will only be used to calculate the distance between
the customer and the seller.

In [36]:
geolocation = pd.read_sql(
    """
    SELECT
        geolocation_zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        geolocation_city,
        geolocation_state
    FROM geolocation;
    """,
    engine
)

geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545622,-46.639294,sao paulo,SP
1,1046,-23.546082,-46.644820,sao paulo,SP
2,1046,-23.546130,-46.642952,sao paulo,SP
3,1041,-23.544392,-46.639500,sao paulo,SP
4,1035,-23.541578,-46.641605,sao paulo,SP


In [37]:
print("Rows:", len(geolocation))
print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

Rows: 1000163
Unique ZIP prefixes: 19015


In [38]:
geo_by_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)

geo_by_zip.head()

,geolocation_zip_code_prefix,latitude,longitude
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634978
2,1003,-23.548993,-46.635731
3,1004,-23.549799,-46.634758
4,1005,-23.549456,-46.636733


## 14. Add Customer Coordinates

We connect the customer ZIP prefix to the geographic coordinates.

The coordinates are intermediate values only. Later, we will use them to
calculate one main feature:

`customer_seller_distance_km`

In [39]:
customers_geo = customers.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

customers_geo = customers_geo.rename(
    columns={
        "latitude": "customer_lat",
        "longitude": "customer_lng"
    }
)

customers_geo = customers_geo.drop(
    columns=["geolocation_zip_code_prefix"]
)

customers_geo.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.498489,-47.396929
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,-23.727992,-46.542848
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,-23.531642,-46.656289
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,-23.499702,-46.185233
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.975100,-47.142925


In [40]:
sellers_geo = sellers.merge(
    geo_by_zip,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

sellers_geo = sellers_geo.rename(
    columns={
        "latitude": "seller_lat",
        "longitude": "seller_lng"
    }
)

sellers_geo = sellers_geo.drop(
    columns=["geolocation_zip_code_prefix"]
)

sellers_geo.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893848,-47.061337
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.383437,-46.947927
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909573,-43.177703
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,-23.657242,-46.612831
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.964803,-46.534420


## 15. Build the Order-Seller Relationship

Some orders have more than one seller.

We first create one row for each unique combination of:

`order_id + seller_id`

This prevents the same seller from being counted several times when an
order contains multiple items from that seller.

In [41]:
order_seller_map = (
    items_with_products[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
    .merge(
        sellers_geo,
        on="seller_id",
        how="left",
        validate="many_to_one"
    )
)

order_seller_map.head()

,order_id,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,sao paulo,SP,-23.565096,-46.518566
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG,-22.262584,-46.171124
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP,-20.553624,-47.387359
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR,-22.929384,-53.135873


### Add customer information

Now we add the customer location so we can compare it with the seller location.

In [42]:
order_customer_geo = (
    orders[["order_id", "customer_id"]]
    .merge(
        customers_geo[
            [
                "customer_id",
                "customer_state",
                "customer_lat",
                "customer_lng"
            ]
        ],
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
)

order_seller_geo = order_seller_map.merge(
    order_customer_geo,
    on="order_id",
    how="left",
    validate="many_to_one"
)

order_seller_geo.head()

,order_id,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng,customer_id,customer_state,customer_lat,customer_lng
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,-22.496953,-44.127492,3ce436f183e68e07877b285a838db11a,RJ,-21.762775,-41.309633
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,sao paulo,SP,-23.565096,-46.518566,f6dd3ec061db4e3987629fe6b26e5cce,SP,-20.220527,-50.903424
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG,-22.262584,-46.171124,6489ae5e4333f3693df5ad4372dab6d3,MG,-19.870305,-44.593326
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP,-20.553624,-47.387359,d4eb9395c8c0431ee92fce09860c5a06,SP,-23.089925,-46.611655
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR,-22.929384,-53.135873,58dbd0b2d70206bf40e62cd34e84d795,SP,-23.243402,-46.827614


## 16. Calculate Customer-Seller Distance

We do not use the latitude and longitude columns directly as model features.

Instead, we use them to calculate the distance between the customer and
the seller.

For orders with more than one seller, we will use the average distance
across the sellers.

In [46]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Convert coordinates from degrees to radians
    lat1, lon1, lat2, lon2 = np.radians(
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return 6371 * c

In [47]:
order_seller_geo["seller_distance_km"] = order_seller_geo.apply(
    lambda row: haversine_km(
        row["customer_lat"],
        row["customer_lng"],
        row["seller_lat"],
        row["seller_lng"]
    )
    if pd.notna(row["customer_lat"])
    and pd.notna(row["customer_lng"])
    and pd.notna(row["seller_lat"])
    and pd.notna(row["seller_lng"])
    else pd.NA,
    axis=1
)

## 17. Customer and Seller in the Same State

We also create a simple geographic feature that tells us whether the
customer and seller are in the same state.

For orders with multiple sellers, the value is True only when all sellers
are in the same state as the customer.

In [48]:
order_seller_geo["same_state"] = (
    order_seller_geo["customer_state"]
    == order_seller_geo["seller_state"]
)

### Aggregate seller and geographic information

Now we return to one row per order.


In [49]:
seller_geo_agg = (
    order_seller_geo
    .groupby("order_id")
    .agg(
        customer_seller_distance_km=("seller_distance_km", "mean"),
        customer_seller_same_state=("same_state", "all")
    )
    .reset_index()
)

seller_geo_agg.head()

,order_id,customer_seller_distance_km,customer_seller_same_state
0,00010242fe8c5a6d1ba2dd792cb16214,301.504649,False
1,00018f77f2f0320c557190d7a144bdd3,585.563902,True
2,000229ec398224ef6ca0657da4fc703e,312.343521,True
3,00024acbcdf0a6daa1e931b038114c75,293.168374,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.16351,False


## 18. Read Order Payments

One order can have more than one payment record.

We aggregate the payment information before joining it with the orders.

In [50]:
order_payments = pd.read_sql(
    """
    SELECT
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value
    FROM order_payments;
    """,
    engine
)

order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Main payment type

Some orders can contain more than one payment record.

We keep the payment type with the highest total value as the main payment type.

In [53]:
payments_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        number_of_payments=("payment_sequential", "count"),
        total_payment=("payment_value", "sum"),
        average_installments=("payment_installments", "mean"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_agg.head()

,order_id,number_of_payments,total_payment,average_installments,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2.0,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3.0,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5.0,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2.0,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3.0,3


In [54]:
payment_type_value = (
    order_payments
    .groupby(["order_id", "payment_type"])["payment_value"]
    .sum()
    .reset_index()
)

main_payment_type = (
    payment_type_value
    .sort_values(
        ["order_id", "payment_value"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_agg = payments_agg.merge(
    main_payment_type,
    on="order_id",
    how="left",
    validate="one_to_one"
)

payments_agg.head()

,order_id,number_of_payments,total_payment,average_installments,max_installments,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2.0,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3.0,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5.0,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2.0,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3.0,3,credit_card


In [55]:
print(payments_agg.shape)
print(payments_agg["order_id"].nunique())
print(payments_agg["order_id"].duplicated().sum())

(99440, 6)
99440
0


## 19. Keep Shipping Limit Information

`shipping_limit_date` may be useful later, but we still need to confirm
whether it was available at the exact prediction time.

For now, we keep the information without using it as a final model feature.

In [56]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

shipping_limits = (
    order_items
    .groupby("order_id")
    .agg(
        earliest_shipping_limit_date=("shipping_limit_date", "min"),
        latest_shipping_limit_date=("shipping_limit_date", "max")
    )
    .reset_index()
)

shipping_limits.head()

,order_id,earliest_shipping_limit_date,latest_shipping_limit_date
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-19 09:45:35,2017-09-19 09:45:35
1,00018f77f2f0320c557190d7a144bdd3,2017-05-03 11:05:13,2017-05-03 11:05:13
2,000229ec398224ef6ca0657da4fc703e,2018-01-18 14:48:30,2018-01-18 14:48:30
3,00024acbcdf0a6daa1e931b038114c75,2018-08-15 10:10:18,2018-08-15 10:10:18
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-13 13:57:51,2017-02-13 13:57:51


## 20. Read Order Reviews

We read the reviews table to understand its structure, but we do not add
review information to the ML table.

Reviews happen after the order experience, so using them for prediction
at order creation time would cause data leakage.

In [57]:
order_reviews = pd.read_sql(
    """
    SELECT
        review_id,
        order_id,
        review_score,
        review_comment_title,
        review_comment_message,
        review_creation_date,
        review_answer_timestamp
    FROM order_reviews;
    """,
    engine
)

order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


## 21. Read Product Category Translation

This table only translates product category names from Portuguese to English.

It does not add new information, so it is not joined to the ML table.

In [58]:
category_translation = pd.read_sql(
    """
    SELECT
        product_category_name,
        product_category_name_english
    FROM product_category_name_translation;
    """,
    engine
)

category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## 22. Build the Final ML Table

Now we have all the main order-level information ready.

We start with `orders` and add:

- Customer information
- Aggregated item and product information
- Seller and geographic information
- Aggregated payment information

The final table should contain exactly one row per order.

In [59]:
ml_table = (
    orders
    .merge(
        customers,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        items_agg,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        seller_geo_agg,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payments_agg,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

ml_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_product_weight,total_product_volume,number_of_product_categories,customer_seller_distance_km,customer_seller_same_state,number_of_payments,total_payment,average_installments,max_installments,main_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,500.0,1976.0,1.0,18.576109,True,3.0,38.71,1.0,1.0,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,400.0,4693.0,1.0,851.495057,False,1.0,141.46,1.0,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,420.0,9576.0,1.0,514.410611,False,1.0,179.12,3.0,3.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,450.0,6000.0,1.0,1822.226338,False,1.0,72.20,1.0,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,250.0,11475.0,1.0,29.676608,True,1.0,28.62,1.0,1.0,credit_card


## 23. Final Checks

Before saving the table, we make sure that the joins did not create
duplicate orders.

The main requirement is:

> One row = One order

In [60]:
print("Shape:", ml_table.shape)
print("Unique orders:", ml_table["order_id"].nunique())
print("Duplicate order IDs:", ml_table["order_id"].duplicated().sum())

Shape: (99441, 26)
Unique orders: 99441
Duplicate order IDs: 0


In [61]:
print("Missing values:")

missing_values = ml_table.isna().sum()
display(missing_values[missing_values > 0])

Missing values:


order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
number_of_items                   775
number_of_sellers                 775
total_price                       775
total_freight                     775
total_product_weight              775
total_product_volume              775
number_of_product_categories      775
customer_seller_distance_km      1264
customer_seller_same_state        775
number_of_payments                  1
total_payment                       1
average_installments                1
max_installments                    1
main_payment_type                   1
dtype: int64

In [62]:
print("Final columns:")

for column in ml_table.columns:
    print("-", column)

Final columns:
- order_id
- customer_id
- order_status
- order_purchase_timestamp
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date
- customer_unique_id
- customer_zip_code_prefix
- customer_city
- customer_state
- number_of_items
- number_of_sellers
- total_price
- total_freight
- total_product_weight
- total_product_volume
- number_of_product_categories
- customer_seller_distance_km
- customer_seller_same_state
- number_of_payments
- total_payment
- average_installments
- max_installments
- main_payment_type


## 24. Save the ML Table

The final order-level table is saved as an artifact.

Notebook 2 will read this file and continue from this point.

In [63]:
os.makedirs("../artifacts", exist_ok=True)

artifact_path = "../artifacts/ml_table1.csv"

ml_table.to_csv(
    artifact_path,
    index=False
)

print(f"Artifact saved to: {artifact_path}")

Artifact saved to: ../artifacts/ml_table1.csv


In [64]:
print("Artifact exists:", os.path.exists(artifact_path))
print("Saved rows:", len(ml_table))
print("Saved columns:", len(ml_table.columns))

Artifact exists: True
Saved rows: 99441
Saved columns: 26


## Final Summary

In this notebook, the main Olist tables were loaded from the database and joined into a single dataset for machine learning.

The resulting dataset contains the order information and the related customer, seller, product, payment, and geographic information needed for the later stages.

The final joined dataset was saved as an artifact and used as the input for Notebook 2.